---

# First EDA on Dummy Data

---

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk

from tqdm import trange
from nltk import tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
import warnings
warnings.filterwarnings('ignore')
nltk.download('omw-1.4', quiet=True)

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (17,7)
plt.rcParams['font.size'] = 18

## Loading Data

In [3]:
data = pd.read_csv('data/synthetic_employee_survey_500_same_structure.csv')
data.head(4)

,ID,Startzeit,Fertigstellungszeit,E-Mail,Name,Language,Ich arbeite bei ...,[Organization]: Ich arbeite im Bereich ...,team_effizienz,staerken_nutzen,...,rahmenbedingungen_change,arbeitgeber_empfehlung,arbeitsfreude,teamstimmung,work_life_balance,ueberlastung,Möchtest Du noch etwas mit den Kolleg*innen teilen?,IterationID,RunID,AbteilungsID
0,1,2026-01-02 13:43:00,2026-01-02 13:57:00,anonymous,anonymous,Deutsch,[Organization],IN-AL (Innovationsfeld Alpha),Stimme voll und ganz zu,Teils/Teils,...,Stimme eher zu,Teils/Teils,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme eher zu,"Ehrlich gesagt: Ich weiß genau, woran ich arbe...",26_1,1,90039353
1,2,2026-01-04 15:18:00,2026-01-04 15:31:00,anonymous,anonymous,Deutsch,[Organization],IN-AL (Innovationsfeld Alpha),Stimme eher zu,Teils/Teils,...,Stimme eher zu,Stimme eher zu,Stimme eher zu,Kann ich nicht beurteilen,Teils/Teils,Stimme eher nicht zu,Bringt ja eh nichts. Wissen hängt bei uns an e...,26_1,1,98837590
2,3,2026-01-03 15:23:00,2026-01-03 15:30:00,anonymous,anonymous,Deutsch,[Organization],KE-EP (Kernbereich Epsilon),Stimme eher zu,Teils/Teils,...,Stimme eher zu,Teils/Teils,Stimme eher zu,Stimme eher nicht zu,Teils/Teils,Stimme voll und ganz zu,"Ich finde die Richtung richtig, auch wenn der ...",26_1,1,92276360
3,4,2026-01-01 13:00:00,2026-01-01 13:06:00,anonymous,anonymous,Deutsch,[Organization],IN-AL (Innovationsfeld Alpha),Stimme voll und ganz zu,Stimme eher zu,...,Stimme voll und ganz zu,Stimme eher zu,Stimme eher zu,Stimme eher zu,Stimme voll und ganz zu,Teils/Teils,"Ehrlich, ich kann nicht mehr. Konflikte werden...",26_1,1,94657693


In [5]:
data.shape

(500, 62)

## Column Renaming

For better handling in Python, the original column names have been renamed to short, `snake_case` identifiers. See the full mapping here: [Table](project_setup/column_mapping.md)

**Rationale:**
The original column headers were long, descriptive sentences (survey questions) containing spaces and special characters. While useful for human readability, they are impractical for programmatic use. Renaming them offers several advantages e.g. reduced errors, improved readability, ...

In [4]:
# Remove useless columns

data = data.drop(['ID','E-Mail', 'Name', 'Language'], axis=1)

In [6]:


# 1. Spaltennamen extrem robust bereinigen:
# - Entfernt das unsichtbare Zeichen \u200b
# - Ersetzt alle Whitespaces (Tabs, Zeilenumbrüche, geschützte Leerzeichen \xa0) durch ein normales Leerzeichen
# - Entfernt Leerzeichen an den Rändern (strip)
data.columns = (
    data.columns
    .str.replace(r'\u200b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# 2. Das korrigierte Dictionary
column_mapping = {
    "Startzeit": "start",
    "Fertigstellungszeit": "ende",
    "Ich arbeite bei ...": "organisation",
    "[Organization]: Ich arbeite im Bereich ...": "bereich",
    "Möchtest Du noch etwas mit den Kolleg*innen teilen?": "freitext_kommentar",
    "IterationID": "iteration_id",
    "RunID": "run_id",
    "AbteilungsID": "abteilungs_id"
}

# 3. Spalten umbenennen
data = data.rename(columns=column_mapping)

data.head(3)

,start,ende,organisation,bereich,team_effizienz,staerken_nutzen,zusammenarbeit_team,psychologische_sicherheit,info_fuehrung_veraenderung,produktgestaltung_aktiv,...,rahmenbedingungen_change,arbeitgeber_empfehlung,arbeitsfreude,teamstimmung,work_life_balance,ueberlastung,freitext_kommentar,iteration_id,run_id,abteilungs_id
0,2026-01-02 13:43:00,2026-01-02 13:57:00,[Organization],IN-AL (Innovationsfeld Alpha),Stimme voll und ganz zu,Teils/Teils,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme eher zu,...,Stimme eher zu,Teils/Teils,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme eher zu,"Ehrlich gesagt: Ich weiß genau, woran ich arbe...",26_1,1,90039353
1,2026-01-04 15:18:00,2026-01-04 15:31:00,[Organization],IN-AL (Innovationsfeld Alpha),Stimme eher zu,Teils/Teils,Stimme voll und ganz zu,Stimme voll und ganz zu,Stimme eher nicht zu,Stimme gar nicht zu,...,Stimme eher zu,Stimme eher zu,Stimme eher zu,Kann ich nicht beurteilen,Teils/Teils,Stimme eher nicht zu,Bringt ja eh nichts. Wissen hängt bei uns an e...,26_1,1,98837590
2,2026-01-03 15:23:00,2026-01-03 15:30:00,[Organization],KE-EP (Kernbereich Epsilon),Stimme eher zu,Teils/Teils,Stimme eher zu,Stimme eher zu,Teils/Teils,Teils/Teils,...,Stimme eher zu,Teils/Teils,Stimme eher zu,Stimme eher nicht zu,Teils/Teils,Stimme voll und ganz zu,"Ich finde die Richtung richtig, auch wenn der ...",26_1,1,92276360


In [97]:
for col in data.columns:
    print(col)

start
ende
organisation
bereich
team_effizienz
staerken_nutzen
zusammenarbeit_team
psychologische_sicherheit
info_fuehrung_veraenderung
produktgestaltung_aktiv
weiterentwicklung_foerderung
fehlerkultur_keine_vorwuerfe
gestaltungsspielraum
kundenfokus
businessentscheidungen_richtung
ehrlichkeit_fuehrung
anforderung_faehigkeit_match
feedback_nutzen
eigenverantwortung_entscheidungen
neues_lernen
beitrag_kundennutzen
erwartungsklarheit
einbeziehung_fuehrung
anspruchsvolle_aufgaben
fehlerkultur_massnahmen
veraenderung_nachvollziehbar
prozessverbesserung_kontinuierlich
einfluss_arbeitsmenge
strategie_einfluss
entscheidungsgeschwindigkeit
prozesse_hinterfragen
hilfe_bitten
lob_anerkennung
klare_ziele
faehigkeiten_ausbau
meinungsfreiheit
zusammenarbeit_schnittstellen
veraenderung_vorantreiben
externe_impulse
respekt
tools_technologien
kulturentwicklung_richtung
offene_kommunikation
strategieklarheit
neue_ideen_ausprobieren
effizienz_schnittstellen
beitrag_wettbewerbsfaehigkeit
arbeitsinformati

## Prepare Data: Numeric Coding for Likert Values (Mapping)

In [7]:
scale = data["team_effizienz"].unique()
print(scale)

<StringArray>
[  'Stimme voll und ganz zu',            'Stimme eher zu',
               'Teils/Teils',       'Stimme gar nicht zu',
      'Stimme eher nicht zu', 'Kann ich nicht beurteilen']
Length: 6, dtype: str


In [8]:
# Find out Number of Likert Columns

# List with index and column names
column_list = list(enumerate(data.columns))
print(column_list)

[(0, 'start'), (1, 'ende'), (2, 'organisation'), (3, 'bereich'), (4, 'team_effizienz'), (5, 'staerken_nutzen'), (6, 'zusammenarbeit_team'), (7, 'psychologische_sicherheit'), (8, 'info_fuehrung_veraenderung'), (9, 'produktgestaltung_aktiv'), (10, 'weiterentwicklung_foerderung'), (11, 'fehlerkultur_keine_vorwuerfe'), (12, 'gestaltungsspielraum'), (13, 'kundenfokus'), (14, 'businessentscheidungen_richtung'), (15, 'ehrlichkeit_fuehrung'), (16, 'anforderung_faehigkeit_match'), (17, 'feedback_nutzen'), (18, 'eigenverantwortung_entscheidungen'), (19, 'neues_lernen'), (20, 'beitrag_kundennutzen'), (21, 'erwartungsklarheit'), (22, 'einbeziehung_fuehrung'), (23, 'anspruchsvolle_aufgaben'), (24, 'fehlerkultur_massnahmen'), (25, 'veraenderung_nachvollziehbar'), (26, 'prozessverbesserung_kontinuierlich'), (27, 'einfluss_arbeitsmenge'), (28, 'strategie_einfluss'), (29, 'entscheidungsgeschwindigkeit'), (30, 'prozesse_hinterfragen'), (31, 'hilfe_bitten'), (32, 'lob_anerkennung'), (33, 'klare_ziele'), 

In [9]:

# Dein Mapping
likert_mapping = {
    "Stimme gar nicht zu": 1,
    "Stimme eher nicht zu": 2,
    "Teils/Teils": 3,
    "Stimme eher zu": 4,
    "Stimme voll und ganz zu": 5,
    "Kann ich nicht beurteilen": np.nan,
}

# Spalten mit Index 3 bis 58 (inklusive) auswählen
cols_to_recode = data.columns[3:59]  # 59, weil Slice-Ende exklusiv ist

# Umkodieren
data[cols_to_recode] = data[cols_to_recode].replace(likert_mapping)

In [11]:
data.head(3).T

,0,1,2
start,2026-01-02 13:43:00,2026-01-04 15:18:00,2026-01-03 15:23:00
ende,2026-01-02 13:57:00,2026-01-04 15:31:00,2026-01-03 15:30:00
organisation,[Organization],[Organization],[Organization]
bereich,IN-AL (Innovationsfeld Alpha),IN-AL (Innovationsfeld Alpha),KE-EP (Kernbereich Epsilon)
team_effizienz,5,4,4
...,...,...,...
ueberlastung,4,2,5
freitext_kommentar,"Ehrlich gesagt: Ich weiß genau, woran ich arbe...",Bringt ja eh nichts. Wissen hängt bei uns an e...,"Ich finde die Richtung richtig, auch wenn der ..."
iteration_id,26_1,26_1,26_1
run_id,1,1,1
